# Figure 5B: MagNET-Zero vs DFT shieldings on the DFT8K benchmark

Reproduces Figure 5B: MagNET-Zero gas-phase shielding predictions vs DFT on the DFT8K benchmark (~7,000
organics, untrained). Protons vs WP04/pcSseg-2, carbons vs wB97X-D/pcSseg-2; residual = DFT minus
MagNET.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/dft8k", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import dft8k_residuals
import paths
import fig5b_plots

In [ ]:
DFT8K_HDF5 = paths.dataset_file("dft8k", root=REPO)

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
shieldings = dft8k_residuals.load_shieldings(DFT8K_HDF5)
print("atoms in DFT8K:", len(shieldings["atomic_numbers"]))

## Residual statistics

In [ ]:
for nucleus in ["H", "C"]:
    errors = dft8k_residuals.residuals(shieldings, nucleus)
    stats = dft8k_residuals.residual_stats(errors)
    print(f"{nucleus}: n={stats['n']:,}  RMSE={stats['rmse']:.4f} ppm  MAE={stats['mae']:.4f} ppm  "
          f"95% abs={stats['abs_p95']:.4f} ppm  fraction < 0.1 ppm={stats['frac_below']:.3f}")

## Residual histograms

In [ ]:
for nucleus in ["H", "C"]:
    errors = dft8k_residuals.residuals(shieldings, nucleus)
    stats = dft8k_residuals.residual_stats(errors)
    print(f"{nucleus}: RMSE={stats['rmse']:.3f} ppm  frac within 0.1 ppm={stats['frac_below']:.3f}")
    if nucleus == "H":
        fig5b_plots.plot_dft8k_residual_histogram(errors, figure_path("fig5b_residuals_1H.png"),
                                      bin_width=0.0025, xlim=0.23, grey_box=0.1, x_tick=0.05)
    else:   # 13C: wider window, no +/-0.1 box since that threshold is 1H-specific
        fig5b_plots.plot_dft8k_residual_histogram(errors, None,
                                      bin_width=0.05, xlim=5.0, grey_box=None, x_tick=1.0)

## Extreme-residual callouts

The largest positive and negative ¹H residuals over the full set. The published negative callout (a
sulfonium zwitterion, -1.040 ppm) is not the true global minimum -- see the printed note.

In [ ]:
extreme_max = dft8k_residuals.find_extreme_residual(DFT8K_HDF5, "H", sign="max")
print("largest positive 1H residual:", extreme_max)
fig5b_plots.show_dft8k_molecule(extreme_max["smiles"])

zwitterion = dft8k_residuals.molecule_by_id(DFT8K_HDF5, "H", molecule_id=88779)
print("published sulfonium-zwitterion callout (id 88779):", zwitterion)
fig5b_plots.show_dft8k_molecule(zwitterion["smiles"])

## Functional-group error breakdown

Mean absolute residual for six functional groups (Carbonyls, Amines, Sulfonyl, Pyridines, Furans,
Nitroso), via RDKit SMARTS matching over every molecule's SMILES.

In [ ]:
for nucleus in ["H", "C"]:
    group_errors = dft8k_residuals.functional_group_errors(DFT8K_HDF5, nucleus)
    print(f"--- {nucleus} ---")
    for name, v in group_errors.items():
        print(f"{name:12s} mean|error|={v['mean_abs_error']:.4f} ppm  n_molecules={v['n_molecules']:,}")
    save = figure_path("fig5b_functional_groups_1H.png") if nucleus == "H" else None
    fig5b_plots.plot_dft8k_functional_group_errors(group_errors, nucleus, save)